## Industry rule for this stage

**Rule:** WOE/IV maps and the scaler/encoder must be `fit` on train only and `applied` (not re-fit) on test.

In [1]:
# add project root to the path
import sys
sys.path.append("../")

In [46]:
import warnings
warnings.filterwarnings('ignore')

In [47]:
from notebooks import *

# Check Point 4 - Feature Transfomration

In [48]:
# load the engineered train/test datasets
clean_eng_train_df = pd.read_csv("../data/processed/engineered_clean_train_df.csv")
clean_eng_test_df = pd.read_csv("../data/processed/engineered_clean_test_df.csv")

#### Numerical columns Bininng

In [49]:
# create a working copy of the train dataframe
copy_train_df = clean_eng_train_df.copy()

In [50]:
# function to bin numeric columns into categorical groups and store bin edges
bin_dict = {}

def generate_binning_column(cols_name):

    for col in cols_name:

        if col == "Age":
            copy_train_df['Age_Group'] = pd.cut(
                copy_train_df['Age'],
                bins=[0, 18, 25, 35, 45, 60, np.inf],
                labels=['Very_Young', 'Young_Adult', 'Early_Career', 'Mid_Career', 'Senior', 'Retired'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Age'] = [0, 18, 25, 35, 45, 60, np.inf]

        elif col == 'Income':
            bins = copy_train_df['Income'].quantile([0, 0.33, 0.66, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['Income_Group'] = pd.cut(
                copy_train_df['Income'],
                bins=bins,
                labels=['Low', 'Medium', 'High'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Income'] = bins

        elif col == 'Employment_length':
            bins = copy_train_df['Employment_length'].quantile([0, 0.25, 0.50, 0.75, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['Employment_Length_Group'] = pd.cut(
                copy_train_df['Employment_length'],
                bins=bins,
                labels=['Fresher', 'Mid-Level', 'Senior', 'Very Senior'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Employment_length'] = bins

        elif col == 'Loan_amount':
            bins = copy_train_df['Loan_amount'].quantile([0, 0.33, 0.66, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['Loan_Amount_Group'] = pd.cut(
                copy_train_df['Loan_amount'],
                bins=bins,
                labels=['Low', 'Medium', 'High'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Loan_amount'] = bins

        elif col == 'Interest_rate':
            bins = copy_train_df['Interest_rate'].quantile([0, 0.33, 0.66, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['Interest_Rate_Group'] = pd.cut(
                copy_train_df['Interest_rate'],
                bins=bins,
                labels=['Low', 'Medium', 'High'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Interest_rate'] = bins

        elif col == 'Loan_percent_income':
            bins = copy_train_df['Loan_percent_income'].quantile([0, 0.25, 0.50, 0.75, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf

            copy_train_df['Loan_Percent_Income_Group'] = pd.cut(
                copy_train_df['Loan_percent_income'],
                bins=bins,
                labels=['Low', 'Medium', 'High', 'Very High'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Loan_percent_income'] = bins

        elif col == 'Credit_history_length':
            bins = copy_train_df['Credit_history_length'].quantile([0, 0.33, 0.66, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['Credit_History_Group'] = pd.cut(
                copy_train_df['Credit_history_length'],
                bins=bins,
                labels=['New', 'Fair', 'Good'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['Credit_history_length'] = bins

        elif col == 'age_employment_ratio':
            bins = copy_train_df['age_employment_ratio'].quantile([0, 0.33, 0.66, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['age_employment_ratio_Group'] = pd.cut(
                copy_train_df['age_employment_ratio'],
                bins=bins,
                labels=['Unstable_Career', 'Moderate_Career', 'Stable_Career'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['age_employment_ratio'] = bins

        elif col == 'income_per_credit_year':
            bins = copy_train_df['income_per_credit_year'].quantile([0, 0.33, 0.66, 1.0]).tolist()
            bins[0] = -np.inf
            bins[-1] = np.inf
            copy_train_df['income_per_credit_year_Group'] = pd.cut(
                copy_train_df['income_per_credit_year'],
                bins=bins,
                labels=['Low_Financial_Growth', 'Moderate_Financial_Growth', 'High_Financial_Growth'],
                include_lowest=True
            )
            print(f"{col} binned successfully")
            bin_dict['income_per_credit_year'] = bins

        else:
            print(f"{col} not found in dataset")

In [51]:
# generate binned columns for all applicable features
generate_binning_column(copy_train_df.columns)

Age binned successfully
Income binned successfully
Home_ownership not found in dataset
Employment_length binned successfully
Loan_intent not found in dataset
internal_credit_rating not found in dataset
Loan_amount binned successfully
Interest_rate binned successfully
Loan_percent_income binned successfully
Default not found in dataset
Credit_history_length binned successfully
Loan_status not found in dataset
is_eligible not found in dataset
age_employment_ratio binned successfully
income_per_credit_year binned successfully


WOE (Weight of Evidence) & IV (Information Value)
> - **What it is:** WOE measures how well a category of a feature separates good vs. bad loans: `WOE = ln(%good / %bad)` per bin. IV sums the WOE contribution across all bins of a feature into one score that tells you the feature's overall predictive strength.
> - **Why we use it:** This is the industry-standard way credit-scoring models pick and encode features — it turns categorical/binned variables into a single monotonic numeric score, and IV gives an objective cutoff to keep or drop a feature (instead of guessing from correlation alone).
> - **What we achieved:** `internal_credit_rating` (IV ≈ 0.87) and `Income` (IV ≈ 0.39) came out as the strongest predictors; `Age` (IV ≈ 0.005) and `Credit_history_length` (IV ≈ 0.003) were "Weak Predictor - DELETE", so they were later dropped from the WOE-encoded feature set.


#### Calculate WOE and IV value for checking feature strength

In [52]:
# check loan status counts across interest rate groups
copy_train_df.groupby('Interest_Rate_Group')['Loan_status'].value_counts().unstack().fillna(0)

Loan_status,False,True
Interest_Rate_Group,,
Low,7807,907
Medium,7044,1509
High,5410,3255


In [53]:
def calculate_woe_iv(df, cat_col, target_col):

    # Only categorical check
    if not pd.api.types.is_object_dtype(df[cat_col]) and not pd.api.types.is_categorical_dtype(df[cat_col]):
        raise ValueError("Column must be categorical")

    # Create table
    woe_iv_table = df.groupby(cat_col)[target_col].value_counts().unstack().fillna(0)

    # Rename (assuming 0=Good, 1=Bad)
    woe_iv_table.columns = woe_iv_table.columns.map({False: 'Good', True: 'Bad'})

    total_good = (df[target_col] == False).sum()
    total_bad  = (df[target_col] == True).sum() 

    # Distribution
    woe_iv_table['Good_dist'] = woe_iv_table['Good'] / total_good
    woe_iv_table['Bad_dist'] = woe_iv_table['Bad'] / total_bad

    # Avoid division by zero
    woe_iv_table['Good_dist'] = woe_iv_table['Good_dist'].replace(0, 1e-10)
    woe_iv_table['Bad_dist'] = woe_iv_table['Bad_dist'].replace(0, 1e-10)

    # WOE & IV
    woe_iv_table['WOE'] = np.log(woe_iv_table['Good_dist'] / woe_iv_table['Bad_dist'])
    woe_iv_table['IV'] = (woe_iv_table['Good_dist'] - woe_iv_table['Bad_dist']) * woe_iv_table['WOE']

    iv_value = woe_iv_table['IV'].sum()

    return woe_iv_table, iv_value, total_good, total_bad

In [54]:
# compute WOE-encoded features for each binned column and store IV results
iv_results = {}
woe_maps = {}
woe_features = []

binned_cols = ['Age_Group','Income_Group', 'Home_ownership', 'Employment_Length_Group', 'Loan_intent',
                'internal_credit_rating', 'Loan_Amount_Group', 'Interest_Rate_Group',
                'Loan_Percent_Income_Group', 'Credit_History_Group', 'age_employment_ratio_Group', 'income_per_credit_year_Group']

for col in binned_cols:
    woe_iv_table, iv_value, total_good, total_bad = calculate_woe_iv(copy_train_df, col, 'Loan_status')

    iv_results[col] = {'iv_value': iv_value, 'woe_table': woe_iv_table}
    
    woe_map = woe_iv_table['WOE'].to_dict()
    woe_maps[col] = woe_map

    copy_train_df[col + "_woe"] = copy_train_df[col].map(woe_map)
    woe_features.append(col + "_woe")


In [55]:
# function to interpret and print IV strength for a feature
# IV interpretation function
def check_iv_strength(iv_value, feature_name):
    print(iv_value)
    if iv_value < 0.02:
        print(f"{feature_name}: Weak Predictor - DELETE ")
    elif 0.02 <= iv_value < 0.1:
        print(f"{feature_name}: Medium Predictor - KEEP ")
    else:
        print(f"{feature_name}: Strong Predictor - KEEP ")

In [56]:
check_iv_strength(iv_results['Age_Group']['iv_value'], 'Age')
check_iv_strength(iv_results['Income_Group']['iv_value'], 'Incomee')
check_iv_strength(iv_results['Home_ownership']['iv_value'], 'Home_ownership')
check_iv_strength(iv_results['Employment_Length_Group']['iv_value'], 'Employment_Length_Group')
check_iv_strength(iv_results['Loan_intent']['iv_value'], 'Loan_intent')
check_iv_strength(iv_results['internal_credit_rating']['iv_value'], 'internal_credit_rating')
check_iv_strength(iv_results['Loan_Amount_Group']['iv_value'], 'Loan_amount')
check_iv_strength(iv_results['Interest_Rate_Group']['iv_value'], 'Interest_Rate_Group')
check_iv_strength(iv_results['Loan_Percent_Income_Group']['iv_value'], 'Loan_Percent_Income_Group')
check_iv_strength(iv_results['Credit_History_Group']['iv_value'], 'Credit_History_Group')
check_iv_strength(iv_results['age_employment_ratio_Group']['iv_value'], 'age_employment_ratio')
check_iv_strength(iv_results['income_per_credit_year_Group']['iv_value'], 'income_per_credit_year')

0.005449560483118273
Age: Weak Predictor - DELETE 
0.386764270923431
Incomee: Strong Predictor - KEEP 
0.3835214488349805
Home_ownership: Strong Predictor - KEEP 
0.05700204360643428
Employment_Length_Group: Medium Predictor - KEEP 
0.09256182872854508
Loan_intent: Medium Predictor - KEEP 
0.8734166597608912
internal_credit_rating: Strong Predictor - KEEP 
0.05289824821183857
Loan_amount: Medium Predictor - KEEP 
0.4549005034705802
Interest_Rate_Group: Strong Predictor - KEEP 
0.6110281849213289
Loan_Percent_Income_Group: Strong Predictor - KEEP 
0.0033200263300483613
Credit_History_Group: Weak Predictor - DELETE 
0.04388750058517553
age_employment_ratio: Medium Predictor - KEEP 
0.14305465278621143
income_per_credit_year: Strong Predictor - KEEP 


In [57]:
# check dtypes of the working train dataframe
copy_train_df.dtypes

Age                                  float64
Income                               float64
Home_ownership                        object
Employment_length                    float64
Loan_intent                           object
internal_credit_rating                object
Loan_amount                          float64
Interest_rate                        float64
Loan_percent_income                  float64
Default                               object
Credit_history_length                float64
Loan_status                             bool
is_eligible                             bool
age_employment_ratio                 float64
income_per_credit_year               float64
Age_Group                           category
Income_Group                        category
Employment_Length_Group             category
Loan_Amount_Group                   category
Interest_Rate_Group                 category
Loan_Percent_Income_Group           category
Credit_History_Group                category
age_employ

In [58]:
# convert WOE columns to float type
cols = [col for col in copy_train_df.columns if col.endswith('_woe')]

copy_train_df[cols] = copy_train_df[cols].astype(float)

#### Apply Same Mapping (Test)

In [59]:
# create a working copy of the test dataframe
copy_test_df = clean_eng_test_df.copy()

In [60]:
# function to apply the same binning (using train bin edges) to test data
def apply_binning_test(df):

    df['Age_Group'] = pd.cut(
        df['Age'],
        bins=bin_dict['Age'],
        labels=['Very_Young', 'Young_Adult', 'Early_Career', 'Mid_Career', 'Senior', 'Retired'],
        include_lowest=True
    )

    df['Income_Group'] = pd.cut(
        df['Income'],
        bins=bin_dict['Income'],
        labels=['Low', 'Medium', 'High'],
        include_lowest=True
    )

    df['Employment_Length_Group'] = pd.cut(
        df['Employment_length'],
        bins=bin_dict['Employment_length'],
        labels=['Fresher', 'Mid-Level', 'Senior', 'Very Senior'],
        include_lowest=True
    )

    df['Loan_Amount_Group'] = pd.cut(
        df['Loan_amount'],
        bins=bin_dict['Loan_amount'],
        labels=['Low', 'Medium', 'High'],
        include_lowest=True
    )

    df['Interest_Rate_Group'] = pd.cut(
        df['Interest_rate'],
        bins=bin_dict['Interest_rate'],
        labels=['Low', 'Medium', 'High'],
        include_lowest=True
    )

    df['Loan_Percent_Income_Group'] = pd.cut(
        df['Loan_percent_income'],
        bins=bin_dict['Loan_percent_income'],
        labels=['Low', 'Medium', 'High', 'Very High'],
        include_lowest=True
    )

    df['Credit_History_Group'] = pd.cut(
        df['Credit_history_length'],
        bins=bin_dict['Credit_history_length'],
        labels=['New', 'Fair', 'Good'],
        include_lowest=True
    )

    df['age_employment_ratio_Group'] = pd.cut(
        df['age_employment_ratio'],
        bins=bin_dict['age_employment_ratio'],
        labels=['Unstable_Career', 'Moderate_Career', 'Stable_Career'],
        include_lowest=True
    )

    df['income_per_credit_year_Group'] = pd.cut(
        df['income_per_credit_year'],
        bins=bin_dict['income_per_credit_year'],
        labels=['Low_Financial_Growth', 'Moderate_Financial_Growth', 'High_Financial_Growth'],
        include_lowest=True
    )

    return df

In [61]:
# apply binning to the test dataframe
apply_binning_test(copy_test_df)

,Age,Income,Home_ownership,Employment_length,Loan_intent,internal_credit_rating,Loan_amount,Interest_rate,Loan_percent_income,Default,...,income_per_credit_year,Age_Group,Income_Group,Employment_Length_Group,Loan_Amount_Group,Interest_Rate_Group,Loan_Percent_Income_Group,Credit_History_Group,age_employment_ratio_Group,income_per_credit_year_Group
0,28.0,56461.0,Rent,5.0,Education,E,5400.0,15.680,0.10,Y,...,9410.166667,Early_Career,Medium,Senior,Low,High,Medium,Fair,Moderate_Career,Moderate_Financial_Growth
1,30.0,108000.0,Mortgage,0.0,Debtconsolidation,B,4925.0,10.946,0.05,N,...,12000.000000,Early_Career,High,Fresher,Low,Medium,Low,Good,Unstable_Career,Moderate_Financial_Growth
2,29.0,102000.0,Own,10.0,Medical,B,18000.0,9.330,0.18,N,...,14571.428571,Early_Career,High,Very Senior,High,Low,High,Good,Stable_Career,Moderate_Financial_Growth
3,29.0,45000.0,Rent,4.0,Venture,B,10000.0,12.420,0.22,N,...,6428.571429,Early_Career,Medium,Mid-Level,Medium,Medium,High,Good,Moderate_Career,Low_Financial_Growth
4,31.0,30000.0,Rent,4.0,Debtconsolidation,B,12000.0,10.390,0.40,N,...,3333.333333,Early_Career,Low,Mid-Level,High,Medium,Very High,Good,Moderate_Career,Low_Financial_Growth
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6479,23.0,42000.0,Rent,2.0,Education,A,7000.0,7.880,0.17,N,...,14000.000000,Young_Adult,Low,Fresher,Medium,Low,High,New,Unstable_Career,Moderate_Financial_Growth
6480,23.0,62000.0,Mortgage,8.0,Homeimprovement,C,9000.0,11.030,0.15,N,...,31000.000000,Young_Adult,Medium,Very Senior,Medium,Medium,Medium,New,Stable_Career,High_Financial_Growth
6481,41.0,98004.0,Mortgage,25.0,Debtconsolidation,C,12000.0,13.570,0.12,N,...,6533.600000,Mid_Career,High,Very Senior,High,High,Medium,Good,Stable_Career,Low_Financial_Growth
6482,24.0,50000.0,Mortgage,7.0,Venture,D,1000.0,14.420,0.02,N,...,16666.666667,Young_Adult,Medium,Senior,Low,High,Low,New,Stable_Career,Moderate_Financial_Growth


Applying the rules of copy_train_df woe maps to the copy_test_df

In [62]:
# map WOE values onto the test dataframe using train mappings
for col in woe_maps:
    copy_test_df[col + '_woe'] = copy_test_df[col].map(woe_maps[col])

In [63]:
# check columns of the test dataframe
copy_test_df.columns

Index(['Age', 'Income', 'Home_ownership', 'Employment_length', 'Loan_intent',
       'internal_credit_rating', 'Loan_amount', 'Interest_rate',
       'Loan_percent_income', 'Default', 'Credit_history_length',
       'Loan_status', 'is_eligible', 'age_employment_ratio',
       'income_per_credit_year', 'Age_Group', 'Income_Group',
       'Employment_Length_Group', 'Loan_Amount_Group', 'Interest_Rate_Group',
       'Loan_Percent_Income_Group', 'Credit_History_Group',
       'age_employment_ratio_Group', 'income_per_credit_year_Group',
       'Age_Group_woe', 'Income_Group_woe', 'Home_ownership_woe',
       'Employment_Length_Group_woe', 'Loan_intent_woe',
       'internal_credit_rating_woe', 'Loan_Amount_Group_woe',
       'Interest_Rate_Group_woe', 'Loan_Percent_Income_Group_woe',
       'Credit_History_Group_woe', 'age_employment_ratio_Group_woe',
       'income_per_credit_year_Group_woe'],
      dtype='object')

#### Calculating Final Credit Score

Train Model

In [64]:
# train logistic regression on WOE features and get cross-validated probabilities
model = LogisticRegression()

copy_train_df['prob'] = cross_val_predict(
    model,
    copy_train_df[woe_features],
    copy_train_df['Loan_status'],
    cv=5,
    method='predict_proba'
)[:, 1]

model.fit(copy_train_df[woe_features], copy_train_df['Loan_status'])

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


Predict Probability

In [65]:
# predict probabilities on the test set using the trained model
copy_test_df['prob'] = model.predict_proba(copy_test_df[woe_features])[:, 1]

Credit score scaling (PDO / Base Score / Base Odds)
> - **What it is:** A formula from traditional credit scorecards that converts the model's predicted probability into a human-readable score on a fixed range (here 300–900). `PDO` (Points to Double the Odds) controls how many points it takes to double the good:bad odds, `base_score`/`base_odds` anchor the scale.
> - **Why we use it:** Raw probabilities aren't intuitive for underwriters/business users — converting to a 300–900 score (like a real credit bureau score) makes the model's output interpretable and comparable to industry scorecards.
> - **What we achieved:** Every customer got a `credit_score` between 300–900; low scores (<650) lined up with higher default rates and high scores (>650) with mostly non-defaulters, confirming the scaled score tracks real risk.


Convert To Credit Score (300–900)

In [66]:
PDO = 50
base_score = 600
base_odds = total_good / total_bad

factor = PDO / np.log(2)
offset = base_score - factor * np.log(base_odds)

# Score formula
copy_train_df['credit_score'] = offset - factor * np.log(copy_train_df['prob'] / (1 - copy_train_df['prob']))
copy_test_df['credit_score'] = offset - factor * np.log(copy_test_df['prob'] / (1 - copy_test_df['prob']))

# Clip to 300–900
copy_train_df['credit_score'] = copy_train_df['credit_score'].clip(300, 900)
copy_test_df['credit_score'] = copy_test_df['credit_score'].clip(300, 900)

#### Basic data anlysis insights on final credit score

Filter customers with low credit score and default status

In [67]:
# filter customers with low credit score
(copy_train_df.loc[
    (copy_train_df['credit_score'] < 650),
    ['credit_score', 'Income', 'Loan_status', 'internal_credit_rating', 'Loan_intent', 'Home_ownership', 'Interest_rate']
]).sort_values(by='internal_credit_rating', ascending=False)

,credit_score,Income,Loan_status,internal_credit_rating,Loan_intent,Home_ownership,Interest_rate
7394,300.000000,138000.0,True,G,Homeimprovement,Mortgage,21.270
20927,300.000000,103872.0,True,G,Debtconsolidation,Rent,20.030
17247,300.000000,88000.0,True,G,Personal,Rent,19.660
12541,300.000000,34800.0,True,G,Venture,Rent,19.160
23145,300.000000,47000.0,True,G,Debtconsolidation,Mortgage,13.006
...,...,...,...,...,...,...,...
7367,575.589254,50000.0,True,A,Personal,Rent,8.490
21816,582.260194,33000.0,False,A,Venture,Rent,5.990
16416,615.116220,42000.0,False,A,Medical,Rent,5.790
16415,633.756149,65000.0,False,A,Venture,Rent,10.084


Filter customers with low credit score but no default

In [68]:
# filter customers with high credit score
copy_train_df.loc[
    (copy_train_df['credit_score'] > 650),
    ['credit_score', 'Income', 'Loan_status', 'Loan_intent', 'Home_ownership', 'Interest_rate']
]

,credit_score,Income,Loan_status,Loan_intent,Home_ownership,Interest_rate
0,750.533887,75800.0,False,Personal,Rent,6.54
2,704.840240,53088.0,False,Personal,Rent,6.54
3,825.686951,228000.0,False,Homeimprovement,Mortgage,7.14
5,801.919595,90000.0,False,Personal,Mortgage,11.49
6,712.401055,54036.0,False,Medical,Mortgage,11.36
...,...,...,...,...,...,...
25926,753.749185,54000.0,False,Personal,Mortgage,5.42
25927,661.095641,25000.0,False,Medical,Mortgage,9.32
25928,706.716972,21600.0,False,Education,Mortgage,5.42
25929,807.396196,81000.0,False,Venture,Rent,7.90


In [69]:
print("\nDefault when credit score is above 650")
print(copy_train_df[copy_train_df['credit_score'] > 650].groupby(['Default']).size())

print("\nDefault when credit score is below 650")
print(copy_train_df[copy_train_df['credit_score'] < 650].groupby(['Default']).size())


print("\nLoan status when credit score is above 650")
print(copy_train_df[copy_train_df['credit_score'] > 650].groupby(['Loan_status']).size())

print("\nLoan status when credit score is below 650")
print(copy_train_df[copy_train_df['credit_score'] < 650].groupby(['Loan_status']).size())


Default when credit score is above 650
Default
N    12268
Y     1201
dtype: int64

Default when credit score is below 650
Default
N    9083
Y    3380
dtype: int64

Loan status when credit score is above 650
Loan_status
False    12761
True       708
dtype: int64

Loan status when credit score is below 650
Loan_status
False    7500
True     4963
dtype: int64


In [70]:
# check credit ratings for non-defaulting, approved loans with credit score above 650
demo_df = copy_train_df[['credit_score', 'Default', 'Loan_status', 'internal_credit_rating']][
    (copy_train_df['credit_score'] > 650) &
    (copy_train_df['Default'] == 'N') &
    (copy_train_df['Loan_status'] == False)
]

print(demo_df['internal_credit_rating'].value_counts())

internal_credit_rating
A    6449
B    4181
C    1002
D      39
E       4
Name: count, dtype: int64


In [71]:
# check credit ratings for non-defaulting, approved loans with credit score below 700
demo_df = copy_train_df[['credit_score', 'Default', 'Loan_status', 'internal_credit_rating']][
    (copy_train_df['credit_score'] < 700) &
    (copy_train_df['Default'] == 'N') &
    (copy_train_df['Loan_status'] == False)
]

print(demo_df['internal_credit_rating'].value_counts())

internal_credit_rating
B    4140
A    2549
C    1405
D     588
E     134
F      28
Name: count, dtype: int64


In [72]:
# check available columns
copy_train_df.columns

Index(['Age', 'Income', 'Home_ownership', 'Employment_length', 'Loan_intent',
       'internal_credit_rating', 'Loan_amount', 'Interest_rate',
       'Loan_percent_income', 'Default', 'Credit_history_length',
       'Loan_status', 'is_eligible', 'age_employment_ratio',
       'income_per_credit_year', 'Age_Group', 'Income_Group',
       'Employment_Length_Group', 'Loan_Amount_Group', 'Interest_Rate_Group',
       'Loan_Percent_Income_Group', 'Credit_History_Group',
       'age_employment_ratio_Group', 'income_per_credit_year_Group',
       'Age_Group_woe', 'Income_Group_woe', 'Home_ownership_woe',
       'Employment_Length_Group_woe', 'Loan_intent_woe',
       'internal_credit_rating_woe', 'Loan_Amount_Group_woe',
       'Interest_Rate_Group_woe', 'Loan_Percent_Income_Group_woe',
       'Credit_History_Group_woe', 'age_employment_ratio_Group_woe',
       'income_per_credit_year_Group_woe', 'prob', 'credit_score'],
      dtype='object')

In [73]:
# High Risk Combo
copy_train_df['high_risk_flag'] = (
    (copy_train_df['Loan_percent_income'] > 0.4) & 
    (copy_train_df['credit_score'] < 600) 
).astype(int)

In [74]:
# High Risk Combo
copy_test_df['high_risk_flag'] = (
    (copy_test_df['Loan_percent_income'] > 0.4) & 
    (copy_test_df['credit_score'] < 600)
).astype(int)

In [75]:
# check columns of train and test dataframes
copy_train_df.columns, copy_test_df.columns

(Index(['Age', 'Income', 'Home_ownership', 'Employment_length', 'Loan_intent',
        'internal_credit_rating', 'Loan_amount', 'Interest_rate',
        'Loan_percent_income', 'Default', 'Credit_history_length',
        'Loan_status', 'is_eligible', 'age_employment_ratio',
        'income_per_credit_year', 'Age_Group', 'Income_Group',
        'Employment_Length_Group', 'Loan_Amount_Group', 'Interest_Rate_Group',
        'Loan_Percent_Income_Group', 'Credit_History_Group',
        'age_employment_ratio_Group', 'income_per_credit_year_Group',
        'Age_Group_woe', 'Income_Group_woe', 'Home_ownership_woe',
        'Employment_Length_Group_woe', 'Loan_intent_woe',
        'internal_credit_rating_woe', 'Loan_Amount_Group_woe',
        'Interest_Rate_Group_woe', 'Loan_Percent_Income_Group_woe',
        'Credit_History_Group_woe', 'age_employment_ratio_Group_woe',
        'income_per_credit_year_Group_woe', 'prob', 'credit_score',
        'high_risk_flag'],
       dtype='object'),
 Index([

Here age is weak predictor so we dont need add that in it so we remove it

In [76]:
# bin Age into readable age groups
bins = [0, 18, 25, 35, 45, 60, np.inf]
labels = ['<18', '18-25', '26-35', '36-45', '46-60', '60+']

copy_train_df['age_group'] = pd.cut(copy_train_df['Age'], bins=bins, labels=labels)
copy_test_df['age_group'] = pd.cut(copy_test_df['Age'], bins=bins, labels=labels)

In [77]:
# drop the original Age column now that age_group has been created
copy_train_df.drop(columns=['Age'], inplace=True)
copy_test_df.drop(columns=['Age'], inplace=True)

#### Remove derived WOE and grouped features, keep only original columns

In [78]:
# remove derived WOE and grouped features, keep only original columns
copy_train_df = copy_train_df[copy_train_df.columns[~copy_train_df.columns.str.contains('_Group|_woe|prob')]]

In [79]:
# remove derived WOE and grouped features, keep only original columns
copy_test_df = copy_test_df[copy_test_df.columns[~copy_test_df.columns.str.contains('_Group|_woe|prob')]]

In [80]:
# Save dataframe
copy_train_df.to_csv("../data/processed/train_df_transformed.csv", index=False)
copy_test_df.to_csv("../data/processed/test_df_transformed.csv", index=False)

#### Types Of Features

Numerical Features

In [81]:
# check dtypes of the train dataframe
copy_train_df.dtypes

Income                     float64
Home_ownership              object
Employment_length          float64
Loan_intent                 object
internal_credit_rating      object
Loan_amount                float64
Interest_rate              float64
Loan_percent_income        float64
Default                     object
Credit_history_length      float64
Loan_status                   bool
is_eligible                   bool
age_employment_ratio       float64
income_per_credit_year     float64
credit_score               float64
high_risk_flag               int64
age_group                 category
dtype: object

In [82]:
# convert Default column to boolean and age_group to object dtype
copy_train_df['Default'] = copy_train_df['Default'].map({'Y' : True, 'N' : False})
copy_train_df['age_group'] = copy_train_df['age_group'].astype('object')

copy_test_df['Default'] = copy_test_df['Default'].map({'Y' : True, 'N' : False})
copy_test_df['age_group'] = copy_test_df['age_group'].astype('object')

In [83]:
# map internal credit rating letters to numeric scale
rating_map = {'A':7, 'B':6, 'C':5, 'D':4, 'E':3, 'F':2, 'G':1}

copy_train_df['internal_credit_rating'] = copy_train_df['internal_credit_rating'].map(rating_map)

copy_test_df['internal_credit_rating'] = copy_test_df['internal_credit_rating'].map(rating_map)

In [84]:
# select all numerical features
train_num_features = copy_train_df.select_dtypes(include=['number']).columns.tolist()

# print total number of numerical features
print('Num of Numerical Features :', len(train_num_features))

# for test_df
test_num_features = train_num_features

Num of Numerical Features : 11


Categorical Features

In [85]:
# select all categorical features
train_cat_features = copy_train_df.select_dtypes(include=['object', 'bool']).columns.tolist()

# print total number of categorical features
print('Num of Categorical Features :', len(train_cat_features))

# for test df
test_cat_features = train_cat_features

Num of Categorical Features : 6


Bool Features

In [86]:
# identify discrete numerical features (low unique values)
train_bool_features = [feature for feature in train_cat_features if len(copy_train_df[feature].unique()) <= 2]

# print total number of discrete features
print('Num of Discrete Features :', len(train_bool_features))

# for test df
test_discrete_features = train_bool_features

Num of Discrete Features : 3


In [87]:
# check identified feature groups
train_num_features, train_cat_features, train_bool_features

(['Income',
  'Employment_length',
  'internal_credit_rating',
  'Loan_amount',
  'Interest_rate',
  'Loan_percent_income',
  'Credit_history_length',
  'age_employment_ratio',
  'income_per_credit_year',
  'credit_score',
  'high_risk_flag'],
 ['Home_ownership',
  'Loan_intent',
  'Default',
  'Loan_status',
  'is_eligible',
  'age_group'],
 ['Default', 'Loan_status', 'is_eligible'])

#### Outlier features

In [88]:
# get all numeric features from dataframe
train_numeric_features = copy_train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# list of features identified with outliers
train_outlier_features = [
    "Income",
    "Employment_length",
    "Loan_amount",
    "Loan_percent_income",
    "Credit_history_length",
    'age_employment_ratio',
    "income_per_credit_year",
    "credit_score",
]

test_outlier_features = train_outlier_features

# remove outlier features from numeric feature list
train_numeric__without_outlier_features = [x for x in train_numeric_features if x not in train_outlier_features]

test_numeric__without_outlier_features = train_numeric__without_outlier_features

In [89]:
# compare numeric feature lists with and without outlier features
train_num_features, train_numeric__without_outlier_features

(['Income',
  'Employment_length',
  'internal_credit_rating',
  'Loan_amount',
  'Interest_rate',
  'Loan_percent_income',
  'Credit_history_length',
  'age_employment_ratio',
  'income_per_credit_year',
  'credit_score',
  'high_risk_flag'],
 ['internal_credit_rating', 'Interest_rate', 'high_risk_flag'])

In [90]:
# check for any columns not covered by the feature groups above
all_covered = (train_numeric__without_outlier_features + train_outlier_features + train_cat_features)
missing = set(copy_train_df.columns) - set(all_covered)
print("Missing columns (will be dropped):", missing)

Missing columns (will be dropped): set()


#### Split X and y

In [91]:
# split dataset into features (X) and target (y)
X = copy_train_df.drop(['Loan_status','credit_score'], axis=1)
y = copy_train_df['Loan_status']

In [92]:
# split into train/validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=32)

#### Preprocessig the columns

In [93]:
# identify numeric columns (excluding categorical features)
numeric_cols = [col for col in X_train.columns if col not in train_cat_features]

In [94]:
categorical_cols = [
    col for col in train_cat_features
    if not (col.startswith('Loan_stat'))
]

# pipeline for categorical features (one-hot encoding)
cat_pipeline_onehot_encoding = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# combine all pipelines into one preprocessor
preprocessor_for_encoding = ColumnTransformer(
    [
        ("encoding_feature_pipeline", cat_pipeline_onehot_encoding, categorical_cols),
        ("numeric_passthrough", "passthrough", numeric_cols)
    ]
)

# fit and transform training data
X_train_prep = preprocessor_for_encoding.fit_transform(X_train)

# transform test data
X_test_prep = preprocessor_for_encoding.transform(X_test)

# transform on test data
copy_test_df_prep = preprocessor_for_encoding.transform(copy_test_df)

#### Renaming the columns

In [95]:
# get feature names after preprocessing
columns = preprocessor_for_encoding.get_feature_names_out()

# convert transformed array into dataframe
scaled_data_X_train = pd.DataFrame(X_train_prep, columns=columns)
scaled_data__copy_test_df = pd.DataFrame(copy_test_df_prep, columns=columns)
scaled_data_X_test = pd.DataFrame(X_test_prep, columns=columns)


# for raw data renaming encoded columns
scaled_data_X_train.columns = scaled_data_X_train.columns.str.replace("encoding_feature_pipeline__", "")
scaled_data__copy_test_df.columns = scaled_data__copy_test_df.columns.str.replace("encoding_feature_pipeline__", "")
scaled_data_X_test.columns = scaled_data_X_test.columns.str.replace("encoding_feature_pipeline__", "")


# for raw data renaming numeric_passthrough columns
scaled_data_X_train.columns = scaled_data_X_train.columns.str.replace("numeric_passthrough__", "")
scaled_data__copy_test_df.columns = scaled_data__copy_test_df.columns.str.replace("numeric_passthrough__", "")
scaled_data_X_test.columns = scaled_data_X_test.columns.str.replace("numeric_passthrough__", "")


In [96]:
# check columns of the transformed dataframes
scaled_data__copy_test_df.columns, scaled_data_X_train.columns

(Index(['Home_ownership_Mortgage', 'Home_ownership_Other', 'Home_ownership_Own',
        'Home_ownership_Rent', 'Loan_intent_Debtconsolidation',
        'Loan_intent_Education', 'Loan_intent_Homeimprovement',
        'Loan_intent_Medical', 'Loan_intent_Personal', 'Loan_intent_Venture',
        'Default_False', 'Default_True', 'is_eligible_False',
        'is_eligible_True', 'age_group_18-25', 'age_group_26-35',
        'age_group_36-45', 'age_group_46-60', 'age_group_60+', 'Income',
        'Employment_length', 'internal_credit_rating', 'Loan_amount',
        'Interest_rate', 'Loan_percent_income', 'Credit_history_length',
        'age_employment_ratio', 'income_per_credit_year', 'high_risk_flag'],
       dtype='object'),
 Index(['Home_ownership_Mortgage', 'Home_ownership_Other', 'Home_ownership_Own',
        'Home_ownership_Rent', 'Loan_intent_Debtconsolidation',
        'Loan_intent_Education', 'Loan_intent_Homeimprovement',
        'Loan_intent_Medical', 'Loan_intent_Personal', 'Loa

In [97]:
categorical_cols_clean = [str(x) for x in categorical_cols]

with open("../artifacts/features/categorical_cols.json", "w") as f:
    json.dump(categorical_cols_clean, f, indent=2)


In [98]:
train_cat_features_raw = [str(x) for x in train_cat_features]

with open("../artifacts/features/train_cat_features.json", "w") as f:
    json.dump(train_cat_features_raw, f, indent=2)

In [99]:
np.savez('../artifacts/splits/x_y_splits.npz', 
                X_train_prep=X_train_prep,
                X_test_prep=X_test_prep,
                y_train=y_train,
                y_test=y_test)

In [100]:
with open("../artifacts/models/preprocessor_for_encoding.pkl", "wb") as f:
    pickle.dump(preprocessor_for_encoding, f)

In [101]:
# Save dataframes
scaled_data_X_train.to_csv("../data/processed/scaled_data_X_train.csv", index=False)
scaled_data__copy_test_df.to_csv("../data/processed/scaled_data__copy_test_df.csv", index=False)
scaled_data_X_test.to_csv("../data/processed/scaled_data_X_test.csv", index=False)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)